# March Madness 2026: Player Prop Betting — Loading & Cleaning

*The data pipeline behind the reporting — IRE 2026*

**Cat Murphy** · June 19, 2026

[IRE 2026 resource page](https://catelizabethmurphy.github.io/ire26/) · [GitHub repo](https://github.com/catelizabethmurphy/ire26)

# What this document is

This is the **loading-and-cleaning pipeline** behind my March Madness player prop betting reporting. A player prop is a bet on one player's stat line — "Cameron Boozer over 22.5 points," for example. During the tournament, thousands of opportunities to risk money on individual college players were made available. None of that data is downloadable, though, so I scraped it; and every site describes the same players, teams, and bets differently, so most of the work is **reconciling** them into one tidy table where one row is one prop line from one operator.

**This public version runs on a one-game sample of prop bets during March Madness** **(Duke vs. Siena, 1st Round)**. There are countless ways to expand this out, though, and I discuss this more in [`../METHODOLOGY.md`](../METHODOLOGY.md).

This version also only includes one primary source of betting data: RotoWire. Now, RotoWire offers several iterations of this data; one is fully public and scrapable, others require paid subscriptions. To illustrate the discrepancies, my sample data folder includes both public and paid data. That folder also includes the ESPN schedule data for the game between Duke and Siena, as this includes information that the RotoWire data does not: seeding, location, accurate times, etc. This is less important for a one-game analysis, but at scale having consistent game information is crucial for merging different datasets.

# Packages

In [ ]:
import re, unicodedata
import pandas as pd

# read the sample data straight from the public repo (works in Colab as-is).
# on a local clone, set:  BASE = "sample_data"
BASE = "https://raw.githubusercontent.com/catelizabethmurphy/ire26/main/sample_analysis/sample_data"

# python stand-ins for the janitor / stringr helpers the R notebook gets from its packages
def clean_names(df):
    df = df.copy()
    df.columns = [re.sub(r"[^0-9a-zA-Z]+", "_", str(c)).strip("_").lower() for c in df.columns]
    return df

def squish(s):
    return re.sub(r"\s+", " ", str(s)).strip()

# What we're looking at

In [ ]:
schedules = pd.read_csv(f"{BASE}/raw_espn_schedule_duke_siena.csv")

scraped_rotowire_props = pd.read_csv(f"{BASE}/raw_rotowire_public_data_duke_siena.csv")

subscription_rotowire_props = pd.read_csv(f"{BASE}/raw_rotowire_subscription_data_duke_siena.csv")

# Standardization functions

As you can see from the data we just looked at, we have a lot of duplicate rows. However, many of these rows aren't identical — usually because of how the team is written (SIE vs. Siena), how the player's name is spelled (A.J. vs. AJ) or how the prop market is referred to (pts vs. points). For that, we're going to write some standardization functions so that we can eliminate those duplicates later. We're also going to create unique IDs for each prop and classify what type of platform it was offered on.

### Standardize team names

In [ ]:
# function to standardize team names in rotowire props and espn schedules
def standardize_team_names(team_name):
    if pd.isna(team_name): return team_name
    return re.sub(r"@", "", str(team_name))

### Standardize team abbreviations

In [ ]:
# function to standardize team abbreviations in espn spreads
def match_school_abbreviations(abbreviation):
    return {"DUKE": "Duke", "SIE": "Siena"}.get(abbreviation, pd.NA)

### Standardize player names

In [ ]:
def standardize_player_names(player_name):
    if pd.isna(player_name): return player_name
    player_name = re.sub(r",? Jr\.?", "", str(player_name))
    player_name = re.sub(r" II(I?)", "", player_name)
    player_name = re.sub(r"\.", "", player_name)
    player_name = unicodedata.normalize("NFKD", player_name).encode("ascii", "ignore").decode("ascii")
    player_name = re.sub(r'".*?"', "", player_name)
    return squish(player_name)

### Standardize prop market names

In [ ]:
def standardize_market_name(market_name):
    if pd.isna(market_name): return market_name
    market_name = re.sub(r"Over/Under", "", str(market_name))
    market_name = market_name.lower()
    market_name = squish(market_name)
    market_name = re.sub(r"\s?\+\s?", "_", market_name)
    market_name = re.sub(r"pts", "points", market_name)
    market_name = re.sub(r"3pt made", "three_pointers_made", market_name)
    market_name = re.sub(r"reb(?![A-Za-z])", "rebounds", market_name)
    market_name = re.sub(r"ast", "assists", market_name)
    market_name = re.sub(r"blk", "blocks", market_name)
    market_name = re.sub(r"stl", "steals", market_name)
    market_name = re.sub(r"-", "_", market_name)
    market_name = re.sub(r"\s", "_", market_name)
    return market_name

### Classify platform types

In [ ]:
def classify_platform_type(platform_name):
    if platform_name in ("betmgm","thescore","fanduel","draftkings","hardrock","caesars","sugarhouse","betrivers","partycasino"):
        return "sportsbook"
    if platform_name in ("sleeper","prizepicks","underdog","pick6","fliff"):
        return "dfs"
    return "other"

### Standardize unique prop ID

In [ ]:
def standardize_id(prop_id):
    prop_id = re.sub(r"\s+", "_", str(prop_id))
    prop_id = re.sub(r"\.", "", prop_id)
    prop_id = re.sub(r"'", "", prop_id)
    prop_id = re.sub(r"-", "", prop_id)
    prop_id = re.sub(r":", "", prop_id)
    prop_id = re.sub(r"\(", "", prop_id)
    prop_id = re.sub(r"\)", "", prop_id)
    return prop_id

# Clean the schedule

In [ ]:
# load and clean schedules
schedules = clean_names(pd.read_csv(f"{BASE}/raw_espn_schedule_duke_siena.csv"))
gd = schedules["game_date"].str.split(", ", expand=True)
schedules["game_day"], schedules["game_date"], schedules["game_year"] = gd[0], gd[1], gd[2]
schedules["game_date"] = pd.to_datetime(schedules["game_date"] + " " + schedules["game_year"], format="%B %d %Y").dt.date
ven = schedules["venue"].str.split(", ", expand=True)
schedules["venue"], schedules["city"], schedules["state"] = ven[0], ven[1], ven[2]
schedules["game"] = schedules[["home_team","away_team"]].min(axis=1) + " vs. " + schedules[["home_team","away_team"]].max(axis=1)
schedules["favored_team"] = schedules["spread_team"].map(match_school_abbreviations)
schedules = schedules.rename(columns={"spread_line": "game_points_spread"})
schedules = schedules[["game","game_date","game_time","tournament","round","region","away_team","away_team_seed","home_team","home_team_seed","venue","city","state","tv_network","game_points_spread","favored_team"]].drop_duplicates("game")
schedules

# Load and clean the prop feeds

### RotoWire paid (subscription) feed — long shape, and the home of the DFS apps

In [ ]:
df = clean_names(pd.read_csv(f"{BASE}/raw_rotowire_subscription_data_duke_siena.csv")).drop_duplicates()
df["data_source"] = "rotowire_subscription"
df["file_date"] = pd.to_datetime(df["file_name"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0]).dt.date
df = df.rename(columns={"player":"player_name","team":"player_team","opponent":"opponent_team","site":"prop_platform","market_name":"prop_market","line":"prop_line"})
df["prop_platform"] = df["prop_platform"].str.replace("-sb", "", regex=False)
df["player_team"] = df["player_team"].str.replace("@", "", regex=False).map(standardize_team_names)
df["opponent_team"] = df["opponent_team"].str.replace("@", "", regex=False).map(standardize_team_names)
df["prop_market"] = df["prop_market"].map(standardize_market_name)
df["prop_platform_type"] = df["prop_platform"].map(classify_platform_type)
df["player_name"] = df["player_name"].map(standardize_player_names)
df["game"] = df[["player_team","opponent_team"]].min(axis=1) + " vs. " + df[["player_team","opponent_team"]].max(axis=1)
df = df.merge(schedules, on="game", how="left")
df = df[df["tournament"] == "NCAA Tournament"].copy()
df["player_team_seed"] = df.apply(lambda r: r["home_team_seed"] if r["player_team"]==r["home_team"] else (r["away_team_seed"] if r["player_team"]==r["away_team"] else pd.NA), axis=1)
df["opponent_team_seed"] = df.apply(lambda r: r["home_team_seed"] if r["opponent_team"]==r["home_team"] else (r["away_team_seed"] if r["opponent_team"]==r["away_team"] else pd.NA), axis=1)
df["player_team_favored"] = df.apply(lambda r: True if r["player_team"]==r["favored_team"] else (False if r["opponent_team"]==r["favored_team"] else pd.NA), axis=1)
def mkid(r):
    s = "_".join("" if pd.isna(x) else str(x) for x in [r["player_name"], r["prop_platform"], r["prop_market"], "no", r["player_team_seed"], r["player_team"], "vs", "no", r["opponent_team_seed"], r["opponent_team"], r["game_date"]])
    return standardize_id(squish(s.lower()))
df["prop_id"] = df.apply(mkid, axis=1)
subscription_rotowire_props = df[["data_source","file_date","tournament","round","game","game_date","player_name","player_team","player_team_seed","opponent_team","opponent_team_seed","game_points_spread","favored_team","player_team_favored","prop_platform","prop_platform_type","prop_market","prop_line","prop_id"]]

subscription_rotowire_props[["player_name","prop_platform","prop_platform_type","prop_market","prop_line","prop_id"]].head(6)

### RotoWire public (scraped) feed — wide shape, pivoted to one row per book

In [ ]:
df = clean_names(pd.read_csv(f"{BASE}/raw_rotowire_public_data_duke_siena.csv")).drop(columns=["scrape_date","scrape_time"])
df["data_source"] = "scraped_rotowire"
df["file_date"] = pd.to_datetime(df["file_name"].str.extract(r"(\d{4}-\d{2}-\d{2})")[0]).dt.date
df = df.rename(columns={"team":"player_team","opponent":"opponent_team","stat_type":"prop_market","mgm_line":"betmgm_line"})
df["prop_market"] = df["prop_market"].map(standardize_market_name)
df["player_name"] = df["player_name"].map(standardize_player_names)
df["player_team"] = df["player_team"].map(standardize_team_names)
df["opponent_team"] = df["opponent_team"].map(standardize_team_names)
df["game"] = df[["player_team","opponent_team"]].min(axis=1) + " vs. " + df[["player_team","opponent_team"]].max(axis=1)
df = df.merge(schedules, on="game", how="left")
df = df[df["tournament"] == "NCAA Tournament"].copy()
df["player_team_seed"] = df.apply(lambda r: r["home_team_seed"] if r["player_team"]==r["home_team"] else (r["away_team_seed"] if r["player_team"]==r["away_team"] else pd.NA), axis=1)
df["opponent_team_seed"] = df.apply(lambda r: r["home_team_seed"] if r["opponent_team"]==r["home_team"] else (r["away_team_seed"] if r["opponent_team"]==r["away_team"] else pd.NA), axis=1)
df["player_team_favored"] = df.apply(lambda r: True if r["player_team"]==r["favored_team"] else (False if r["opponent_team"]==r["favored_team"] else pd.NA), axis=1)
line_cols = [c for c in df.columns if c.endswith("_line")]
df = df.melt(id_vars=[c for c in df.columns if c not in line_cols], value_vars=line_cols, var_name="prop_platform", value_name="prop_line")
df["prop_platform"] = df["prop_platform"].str.replace("_line", "", regex=False)
df = df[df["prop_line"].notna()].copy()
df["prop_platform_type"] = df["prop_platform"].map(classify_platform_type)
def mkid(r):
    s = "_".join("" if pd.isna(x) else str(x) for x in [r["player_name"], r["prop_platform"], r["prop_market"], "no", r["player_team_seed"], r["player_team"], "vs", "no", r["opponent_team_seed"], r["opponent_team"], r["game_date"]])
    return standardize_id(squish(s.lower()))
df["prop_id"] = df.apply(mkid, axis=1)
scraped_rotowire_props = df[["data_source","file_date","tournament","round","game","game_date","player_name","player_team","player_team_seed","opponent_team","opponent_team_seed","game_points_spread","favored_team","player_team_favored","prop_platform","prop_platform_type","prop_market","prop_line","prop_id"]]

scraped_rotowire_props[["player_name","prop_platform","prop_platform_type","prop_market","prop_line","prop_id"]].head(6)

# Bind into one table

Stack the feeds, then collapse on `prop_id` so the same bet seen in more than one feed (or more than one scrape) becomes one row — keeping a count of how many times it was identified (`number_times_identified`) and which sources and files saw it, so any line is traceable.

In [ ]:
keep = ["data_source","file_date","tournament","round","game","game_date","player_name","player_team","player_team_seed","opponent_team","opponent_team_seed","game_points_spread","favored_team","player_team_favored","prop_platform","prop_platform_type","prop_market","prop_line","prop_id"]
all_props = pd.concat([subscription_rotowire_props[keep], scraped_rotowire_props[keep]], ignore_index=True)
agg = all_props.groupby("prop_id", sort=False).agg(
    number_times_identified=("prop_id", "size"),
    data_source=("data_source", lambda s: ", ".join(dict.fromkeys(s.astype(str)))),
    file_date=("file_date", lambda s: ", ".join(dict.fromkeys(s.astype(str)))),
).reset_index()
all_props = (all_props.drop_duplicates("prop_id", keep="first")
             .drop(columns=["data_source","file_date"])
             .merge(agg, on="prop_id"))
all_props = all_props[["tournament","round","game","game_date","player_name","player_team","player_team_seed","opponent_team","opponent_team_seed","game_points_spread","favored_team","player_team_favored","prop_platform","prop_platform_type","prop_market","prop_line","prop_id","number_times_identified","data_source","file_date"]]

all_props

# Explore the results

In [ ]:
pd.DataFrame([{
    "n_props": all_props["prop_id"].nunique(),
    "n_props_siena": all_props.loc[all_props["player_team"].str.contains("Siena"), "prop_id"].nunique(),
    "n_props_duke": all_props.loc[all_props["player_team"].str.contains("Duke"), "prop_id"].nunique(),
    "n_players": all_props["player_name"].nunique(),
    "n_markets": all_props["prop_market"].nunique(),
    "n_platforms": all_props["prop_platform"].nunique(),
}])

---

See [`../METHODOLOGY.md`](../METHODOLOGY.md) for more on this project.